# Case Study: Multi-Method Sentiment Analysis of Literary Texts

In this case study, we apply **multiple sentiment analysis methods** to two classic literary works and compare their results:

- **Text A** — *Alice's Adventures in Wonderland* (Lewis Carroll) — a whimsical, emotionally varied narrative
- **Text K** — *The Metamorphosis* (Franz Kafka) — a dark, psychologically intense novella

## Methods Compared

1. **VADER** — Rule-enhanced lexicon approach
2. **TextBlob** — Pattern-based lexicon approach
3. **Sentence-BERT** — Transformer-based embedding similarity

We examine both the **cumulative sentiment arc** (how sentiment evolves across the text) and **aggregate statistics** for each method.

In [ ]:
# pip install vaderSentiment textblob sentence-transformers matplotlib numpy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Data paths ---
A_path = '../Data/A.txt'   # Alice in Wonderland
K_path = '../Data/K.txt'   # The Metamorphosis

def prepare_data(file_path):
    """Load and filter lines from a text file."""
    with open(file_path, 'r') as f:
        lines = f.readlines()
    lines = [line.strip() for line in lines]
    lines = [line for line in lines if len(line) > 10]
    return lines

A_lines = prepare_data(A_path)
K_lines = prepare_data(K_path)

print(f"Alice in Wonderland: {len(A_lines)} lines")
print(f"The Metamorphosis:   {len(K_lines)} lines")

## 1. VADER Sentiment Analysis

VADER is designed for **sentence-level** analysis. It uses a lexicon of sentiment-rated words combined with grammatical rules (e.g., negation, intensifiers, punctuation) to produce a compound score in [-1, 1].

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

def vader_scores(lines):
    """Return per-line compound scores using VADER."""
    return [vader.polarity_scores(line)['compound'] for line in lines]

A_vader = vader_scores(A_lines)
K_vader = vader_scores(K_lines)

print(f"Alice — mean compound: {np.mean(A_vader):.4f}, std: {np.std(A_vader):.4f}")
print(f"Kafka — mean compound: {np.mean(K_vader):.4f}, std: {np.std(K_vader):.4f}")

## 2. TextBlob Sentiment Analysis

TextBlob provides two scores per text: **polarity** [-1, 1] and **subjectivity** [0, 1]. We focus on polarity for comparison with VADER.

In [ ]:
from textblob import TextBlob

def textblob_scores(lines):
    """Return per-line polarity scores using TextBlob."""
    return [TextBlob(line).sentiment.polarity for line in lines]

A_textblob = textblob_scores(A_lines)
K_textblob = textblob_scores(K_lines)

print(f"Alice — mean polarity: {np.mean(A_textblob):.4f}, std: {np.std(A_textblob):.4f}")
print(f"Kafka — mean polarity: {np.mean(K_textblob):.4f}, std: {np.std(K_textblob):.4f}")

## 3. Sentence-BERT Cosine Similarity

We use SBERT to embed each line, then measure cosine similarity to positive and negative reference phrases. The difference (positive_sim − negative_sim) serves as a sentiment score.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

sbert = SentenceTransformer('all-MiniLM-L6-v2')

# Reference phrases for positive/negative sentiment
refs = sbert.encode([
    "This is joyful, wonderful, happy, and delightful.",
    "This is terrible, sad, painful, and depressing."
])

def sbert_scores(lines):
    """Return per-line sentiment scores using SBERT cosine similarity."""
    embeddings = sbert.encode(lines, show_progress_bar=True, batch_size=64)
    pos_sim = cosine_similarity(embeddings, refs[0:1]).flatten()
    neg_sim = cosine_similarity(embeddings, refs[1:2]).flatten()
    return (pos_sim - neg_sim).tolist()

print("Encoding Alice in Wonderland...")
A_sbert = sbert_scores(A_lines)
print("Encoding The Metamorphosis...")
K_sbert = sbert_scores(K_lines)

print(f"\nAlice — mean SBERT score: {np.mean(A_sbert):.4f}, std: {np.std(A_sbert):.4f}")
print(f"Kafka — mean SBERT score: {np.mean(K_sbert):.4f}, std: {np.std(K_sbert):.4f}")

## 4. Cumulative Sentiment Arcs — Comparison

The cumulative sum of per-line scores reveals how overall sentiment builds across the narrative. We plot all three methods side-by-side for each text.

In [ ]:
def plot_cumulative_arcs(lines_scores_dict, title):
    """
    Plot cumulative sentiment arcs for multiple methods.
    
    Args:
        lines_scores_dict: dict of {method_name: list_of_scores}
        title: plot title
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
    colors = ['#2196F3', '#FF9800', '#4CAF50']
    
    for ax, (method, scores), color in zip(axes, lines_scores_dict.items(), colors):
        cumulative = np.cumsum(scores)
        ax.plot(range(len(cumulative)), cumulative, color=color, alpha=0.8, linewidth=0.8)
        ax.set_title(method, fontsize=13)
        ax.set_xlabel('Line Number')
        ax.set_ylabel('Cumulative Score')
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.4)
        ax.grid(True, alpha=0.2)
    
    fig.suptitle(title, fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Alice in Wonderland
plot_cumulative_arcs(
    {'VADER': A_vader, 'TextBlob': A_textblob, 'SBERT': A_sbert},
    'Sentiment Arc — Alice in Wonderland'
)

# The Metamorphosis
plot_cumulative_arcs(
    {'VADER': K_vader, 'TextBlob': K_textblob, 'SBERT': K_sbert},
    'Sentiment Arc — The Metamorphosis'
)

## 5. Score Distribution Comparison

Comparing the distribution of per-line scores shows how each method perceives the emotional range of each text.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

texts = {
    'Alice in Wonderland': {'VADER': A_vader, 'TextBlob': A_textblob, 'SBERT': A_sbert},
    'The Metamorphosis':   {'VADER': K_vader, 'TextBlob': K_textblob, 'SBERT': K_sbert}
}
colors = ['#2196F3', '#FF9800', '#4CAF50']

for row, (text_name, methods) in enumerate(texts.items()):
    for col, ((method, scores), color) in enumerate(zip(methods.items(), colors)):
        ax = axes[row][col]
        ax.hist(scores, bins=40, color=color, alpha=0.7, edgecolor='white')
        ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        ax.set_title(f'{method} — {text_name}', fontsize=10)
        ax.set_xlabel('Score')
        ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 6. Positive vs Negative Cumulative Arcs (VADER)

VADER uniquely provides separate positive and negative scores. Plotting both reveals the independent emotional strands in each text.

In [ ]:
def plot_vader_pos_neg(lines, title):
    """Plot cumulative positive and negative VADER scores."""
    pos_scores = [vader.polarity_scores(line)['pos'] for line in lines]
    neg_scores = [vader.polarity_scores(line)['neg'] for line in lines]
    
    plt.figure(figsize=(12, 5))
    plt.plot(np.cumsum(pos_scores), label='Positive', color='green', alpha=0.8, linewidth=0.8)
    plt.plot(np.cumsum(neg_scores), label='Negative', color='red', alpha=0.8, linewidth=0.8)
    plt.xlabel('Line Number')
    plt.ylabel('Cumulative Score')
    plt.title(f'VADER Positive vs Negative — {title}')
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

plot_vader_pos_neg(A_lines, 'Alice in Wonderland')
plot_vader_pos_neg(K_lines, 'The Metamorphosis')

## 7. Summary Statistics

In [ ]:
import pandas as pd

summary_data = []
for text_name, methods in [
    ('Alice in Wonderland', {'VADER': A_vader, 'TextBlob': A_textblob, 'SBERT': A_sbert}),
    ('The Metamorphosis',   {'VADER': K_vader, 'TextBlob': K_textblob, 'SBERT': K_sbert})
]:
    for method, scores in methods.items():
        s = np.array(scores)
        summary_data.append({
            'Text': text_name,
            'Method': method,
            'Mean': f'{s.mean():.4f}',
            'Std': f'{s.std():.4f}',
            'Median': f'{np.median(s):.4f}',
            '% Positive': f'{(s > 0).mean() * 100:.1f}%',
            '% Negative': f'{(s < 0).mean() * 100:.1f}%',
            '% Neutral': f'{(s == 0).mean() * 100:.1f}%',
        })

summary_df = pd.DataFrame(summary_data)
summary_df

## 8. Method Agreement Analysis

How often do the three methods agree on the direction of sentiment for each line?

In [ ]:
def compute_agreement(scores_list):
    """Compute pairwise and unanimous agreement on sentiment direction."""
    # Convert to sign: +1, -1, or 0
    signs = [np.sign(np.array(s)) for s in scores_list]
    n = len(signs[0])
    
    # Unanimous agreement (all 3 same sign, excluding neutral)
    all_agree = sum(
        1 for i in range(n)
        if signs[0][i] == signs[1][i] == signs[2][i] and signs[0][i] != 0
    )
    
    return all_agree / n * 100

alice_agreement = compute_agreement([A_vader, A_textblob, A_sbert])
kafka_agreement = compute_agreement([K_vader, K_textblob, K_sbert])

print(f"Unanimous agreement (all 3 methods, same direction):")
print(f"  Alice in Wonderland: {alice_agreement:.1f}%")
print(f"  The Metamorphosis:   {kafka_agreement:.1f}%")

## Discussion

Key observations from the multi-method comparison:

**Sentiment arc differences:** Each method captures sentiment differently. VADER and TextBlob are purely lexicon-based and respond to individual sentiment-bearing words, while SBERT captures broader semantic context. This often leads to smoother arcs from SBERT.

**Text characteristics:** *Alice in Wonderland* typically shows a more balanced sentiment profile with frequent swings, consistent with its whimsical tone. *The Metamorphosis* tends to skew negative across all methods, reflecting Kafka's darker subject matter.

**Method agreement:** The methods often disagree at the line level, especially on ambiguous or context-dependent lines. This highlights the importance of using multiple approaches and not relying on a single method.

**Neutral handling:** Lexicon-based methods (VADER, TextBlob) produce many zero scores for lines without sentiment-bearing words, while SBERT rarely produces exactly zero — it always finds some semantic similarity to the reference phrases.

### Methodological Note

Literary texts present unique challenges for sentiment analysis tools designed for short informal text (reviews, social media). Archaic language, irony, and narrative context are difficult for all methods. These limitations should be considered when interpreting the results.